# Distant Viewing Tutorial 1: Documerica and Color Analysis

[[**homepage**](https://www.distantviewing.org)]

This notebook provides an introduction to the methods presented in the book
*Distant Viewing: Computational Exploration of Digital Images*
(MIT Press, 2023). We apply them to a small collection of photographs from
Documerica, a project of the U.S. Environmental Protection Agency that
photographed the country during the 1970s. We do not assume any prior
knowledge of Python or computer vision in
these notes. While a complete introduction to Python is not in the scope of our
introduction, we do our best to highlight the main features of the language
as they apply to the application here. Here are the specific learning outcomes
for the tutorial:

1. Explain how digital images are stored as pixel arrays.
2. Connect the structure of digital images with computational methods through
the distant viewing framework.
3. Apply pre-constructed Python code to study a collection of digital images.
4. Explain measurements such as hue, saturation, chroma, and value using
color theory.
5. Compare the use of color in photographs taken in different places.
6. Produce annotations with state-of-the-art computer vision algorithms to
detect faces using the distant viewing toolkit.

For further information about the distant viewing toolkit (**dvt**), the
open-source Python package that we have developed, please see the
[project's homepage](https://github.com/distant-viewing/dvt).
More information about the theory of distant viewing can be the found in our
book, which is
available to download for free under an open access license on our
[website](https://www.distantviewing.org/book/). A second notebook
following up on the methods here using moving images can be found
[here](https://colab.research.google.com/github/distant-viewing/dvt/blob/main/tutorials/Distant_Viewing_Tutorial_2_Shots_and_Faces_in_Moving_Images.ipynb).

## 1.1 Setup

As a first step, we need to install a few additional Python components,
download the collection of photographs, and tell Python all of the functions
that we will need later in the tutorial. To get started, we will use the
code below to install the module called **dvt** (the distant viewing toolkit),
which contains several useful functions specifically designed to apply computer
vision algorithms to collections of humanities data. The exclamation point at
the start of the line of code tells the notebook that we we want to directly run a command line tool outside of Python itself. Here, we are using the tool
called **pip** that can used to install additional functionality for Python.
To run the code, hover your mouse somewhere over the background of the
code. This will show a triangular play button on the left-hand side of the code
block. Hit the button and wait for it to finish, which may take a minute or
two as Colab always takes a bit of time to set up when running the first
code block.

In [ ]:
!pip install -q "dvt[video,face]>=2.0.2"

Next, below we have some code to download the photographs and the metadata that
we are working with in this notebook. These use the program **wget** to download
files from the toolkit's GitHub repository followed by the **tar** command to
unpack the directory containing the photographs. We also download a file of
helper functions called `funs.py` that makes it easier to work with tables of
data, plots, and images. The last line uses the **dvt** command itself to
download the computer vision model that is needed in the final section of the
notebook for finding faces. As above, hover over the code and then click on
the play button on the left-hand side of the code block.

In [ ]:
!wget -q -nc "https://raw.githubusercontent.com/distant-viewing/dvt/main/tutorials/data/documerica.tar"
!wget -q -nc "https://raw.githubusercontent.com/distant-viewing/dvt/main/tutorials/data/documerica.csv"
!wget -q -nc "https://raw.githubusercontent.com/distant-viewing/dvt/main/tutorials/funs.py"
!tar xf documerica.tar
!dvt models fetch --annotators face

In the last part of the setup, we will run some code in Python (note that the
lines below do not start with an exclamation mark). This code here uses
the **import** command to tell Python which libraries and functions we are
going to use in the notebook. Specifically, these are **numpy** for working
with large arrays of numbers, **polars** for working
with tabular datasets, **plotnine** for data visualization, **cv2** (the
OpenCV library) for image processing, the **dvt** module that we described
above, and the helper functions in **funs**. The name `c` is a shorthand for
referring to the columns of a table, which we will use throughout the notebook.

In [ ]:
import numpy as np
import polars as pl
import cv2
import dvt
from plotnine import ggplot, aes
from polars import col as c

import funs
from funs import DSImage

In Google Colab, your working environment resets itself everytime you reopen a
notebook. Therefore, all of the steps above need to be re-run each time that
you start the notebook. If you were running this code on your own machine, the
installation of the **dvt** package and downloading the data would only need to
be done once. Loading the modules in the final code chunk, however, always needs
to be run each time that Python is restarted.

## 1.2 Documerica Dataset

Between 1972 and 1977, the newly created U.S. Environmental Protection Agency
hired freelance photographers to document subjects of environmental concern
across the country. The project was called Documerica. The photographers
recorded pollution, industry, and the land, but also a great deal of everyday
life: people at work, at home, and in the street. The photographs are now held
by the U.S. National Archives and are in the public domain. In this notebook
we work with a sample of 100 of them.

Before we jump into the analysis of the images, it is important
to take a moment to look at the metadata that we have attached to each
photograph
and to understand the structure of the dataset. In the code below, we use the
`read_csv` function from the polars module (which we have given the short
name **pl** following standard Python conventions) to load the csv file that has
one row for each photograph in our dataset. We will save the output of the function
as an object named `photos`. In the final line of the code, we write the object
name all by itself, which causes the first and last few rows of the dataset to be
printed inside of the notebook for us to look at. For each photograph, we
have the identifier given to it by the National Archives, the file name of
the image, the name of the photographer, the year and month that it was taken,
the place, and the caption written at the time.

In [ ]:
photos = pl.read_csv("documerica.csv")
photos

Throughout the notebook we will write code as a chain of steps, with each step
on its own line, starting from a dataset. The code below starts with the
`photos` data, groups together the rows from the same region of the country,
counts the number of photographs in each group, and sorts the result. Two of
the photographs in our sample were taken outside of the United States.

In [ ]:
(
    photos
    .group_by(c.region)
    .agg(count = pl.len())
    .sort(c.count, descending=True)
)

We can also look at a few of the photographs themselves. The function
`DSImage.plot_image_grid` displays the images from a dataset in a grid. Here we
show the first twelve photographs, labelled with the state in which each was
taken.

In [ ]:
(
    photos
    .head(12)
    .pipe(DSImage.plot_image_grid, label_name="state", ncol=6)
)

Now that we have a sense of our data, what kinds of questions might we be
interesed in exploring? Our analyses in the remainder of the notebook will
focus on the use of color in the photographs and on whether or not they show
people.

## 1.3 Digital Images

Now that we have seen the metadata for the photographs, let's look at how
digital images are manipulated in Python by loading in a single
image. All of the photographs are stored as JPEG files (an abbreviation
for the Joint Photographic Experts Group). This is a common image format that
can be opened and understood by nearly any program or device that works with
images. If you opened a JPEG file on your computer or phone, it would display
the image without any special setup required. Many of the images that you see
on public websites are stored as JPEG files and are processed and displayed
by your browser.

In the code below, we use the function `DSImage.read` to load an image into
Python. We save the image as an object named `img`. We have selected a
photograph of one of the Documerica photographers, David Hiser, standing above
the Colorado River at Dead Horse Point in Utah. After loading
the image into Python, we display it with the function `DSImage.show`.

In [ ]:
img = DSImage.read("documerica/545607.jpg")
DSImage.show(img)

We see that it's relatively easy to load a common image file format into Python
and then to display it within the notebook. In order to best understand the
computational analyses that follow, it will be helpful to investigate the
the way that digital image is represented inside of Python. We can use the
built-in Python function `type` to see the obect type of any Python object.
Let's do that below:

In [ ]:
type(img)

You may have reasonably assumed that the object would have a name related
to the fact that it contains an image. But, we see that it is cryptically
called a `numpy.ndarray`. What does this mean? This is a generic data type
created by the numpy library (the same one that we loaded above in the setup
section) to store rectangular blocks of numbers.

To understand how an array of numbers can represent an image, we will print out
the object's shape attribute (an attribute is a characteristic of a Python
object that we can access with the object name followed by a `.` and the
attribute name). This tells us how the numbers in array are arranged.

In [ ]:
img.shape

We see that the shape of the object has three components. The first number tells
us how many rows of numbers there are and the second tells us how many columns
there are. The third number indicates that there is a third dimension with a
size of three. The easiest way to picture this is to think of having three sets
of rectangular grids of numbers, each with 600 rows and 404 columns. Think of
an Excel file with three sheets, each having a grid of numbers of the same
size.

A specific row and column in this grid of numbers represents a *pixel* (picture + element), the
smallest individal component of an image. We need three different numbers for
each pixel to indicate the amount of red, green, and blue light that is needed
to combine to create the color at each particular location. The Python library
that we are using here represents the quantity of light on a scale from 0 (not
turned on at all) through 255 (as bright as possible). Blending these three
components together can recreate nearly any color observable by the human eye.

To make this more concrete, let's see an example of the numbers that create the
image above. There are far too many to look at all at once. Instead, we will use
a bracket notation to select a small block of the image: rows 60 through 67,
columns 300 through 305, and the
first color component. Python uses a convention that is common in computer
programing that starts counting at zero, so the `0` below grabs from the first
array of numbers, which here correspond to the red color intensity.

In [ ]:
img[60:68, 300:306, 0]

The portion of the image that we grabbed above is a small piece of the sky,
to the right of the photographer's head. We see that to represent this part
of the image we need to turn on a lot of red
light (around 190 out a possible 255). However, if we look at the image
there appears to be no red in the sky at all. To understand how this is the
case, let's look at the second
component, which is the amount of green light.

In [ ]:
img[60:68, 300:306, 1]

And while we are at it, also the amount of blue light.

In [ ]:
img[60:68, 300:306, 2]

Looking above, we see that the red, green, and blue lights are all turned on
at a high level in this part of the image. When we blend
light from all three colors together at the same level, we get a shade of grey. This is something closer to black when the colors are all turned low, and something closer to white when
the colors are all turned higher. Here the green and blue are turned on
slightly more than the red, which gives the pale blue that we see in the sky.

To solidify our understanding how these components work, let's see another
part of the image, a single pixel from the rock in the lower left of the
photograph, at row 560 and column 100. The code below shows the red, green,
and blue components that make up its color.

In [ ]:
img[560, 100, :]

The tan color of the rock comes from blending a good amount of red light,
a bit less green, and less blue again. Looking at a color wheel
should help explain why an orange or tan color comes from mixing green with a
larger amount of red.

Different image processing libraries have slightly different conventions for
how to represent digital images. Most use the same ordering of the colors
(red, green, blue), and some use fractions between 0 and 1 rather than integers
between 0 and 255. Certain image formats include a fourth component, called an
alpha channel, to represent image opacity. Other formats contain a single
color channel to represent grayscale images. However, all of these formats
have the same underlying concept of representing digital images through
numbers that indicate pixel intensities. This is a very different way of
thinking about images than the way that humans process visual signals and is
something that we will explore in the next section.

## 1.4 Distant Viewing: Theory

Computers represent images by understanding them as three-dimensional arrays of
numbers. This is very different from the way that images are interpreted and
used by human viewers. Furthermore, the connection between these two
representations is not at all obvious. There is no way to understand what is
being represented by a small subset of pixel intensities without seeing a large
part of the image as a whole. Even something as simple as the amount of blue
light in a given pixel can be difficult to understand. A lot of blue could be
the color blue, or could just be blended with red and green to make white.
So, in order to do computational analyses with large collections of digital
images, we first need to convert these raw pixel intensities into
representations that match the interpretations of the images that we are
interested in studying.

The theory behind distant viewing stems from
this exact realization. Namely, that the way digital images are represented
forces us to construct *annotations* that hold structured data that aligns with
our research questions. These annotations, which can be created manually or
using computational algorithms, are both destructive (there is information lost
in the process of creating them) and open to interpretation (there is never a
neutral way of creating annotations; decisions always need to be made about
how they are created).

We will work towards more complex annotations, but let's start with one of the
most straightforward: image brightness. Pixel intensities tell us how much
to turn on the red, green, and blue lights at each postion of the image to
create a display of a given image. The higher these numbers are, it stands to
reason, the brighter the image will be when shown on a digital display. So,
one way to represent a meaningful annotation about an image is to take the
average value of all pixel intensities. We can do this with the following code,
which uses the numpy function `mean` to compute the average (mean) of all the
values in an array.

In [ ]:
np.mean(img)

Little argument probably needs to be made to convince someone that a lot of
information is lost between this one number and all of the rich information that
is present in the photograph. There is no information
about the man standing in the middle of the frame, the camera around his neck,
the river and the canyon behind him, or the way each of these elements is
arranged in the frame. So, clearly this process of creating annotations is a
destructive one. The difference between the summarized annotation (a single
number) and the information in the original image is known in information
theory as a *semantic gap*. But what about the second part of the theory, that
this measurement is non-netural and represents specific choices about how we
want to *view* at a distance? While this may seem less obvious, there are many
different ways of measuring the brightness of an image. For example, we might
want to consider the median value of the pixel intensities rather than their
average value as in the code below.

In [ ]:
np.median(img)

Alternatively, because the human eye is more sensitive to green than blue or
red light, we might want to weight the brightness more heavily based on the
color of light that is being used. The scans of these photographs include the
dark edge of the film around each image, which influences the brightness of
the image as a whole.
Perhaps we would want to only take the brightness at the middle part of the
image. And of course, why do we even care about the brightness of the image in
the first place? Once we start thinking about all of these different options,
it should become clear that there is no perfect way to represent any element of
an image as structured data. Choices and tradeoffs are always being made.
Eventually we need to make some of those choices and see what we can learn with
them, while keeping the caveats about the nature of image annotations and the
resulting semantic gaps always in the back of our mind. In other words, when we are analyzing images through computer vision, we are distant viewing.

## 1.5 Annotating Image Brightness

We have now carefully worked through the way the digital images are stored,
understood the implication for this in terms of the theory of distant viewing,
and shown one particular way of constructing an annotation through image
brightness. Let's now put this together to do an analysis of the photographs
based on their overall brightness. As a first step, we need to repeat the
process used with the one photograph above to all of the images in the dataset.
In order to do this we use a loop in Python. This consists of the keyword `for`,
followed by a block of indented code. Each of the lines of the indented code
will be run once for every row of the `photos` dataset. So, we will load the
image of each photograph into Python, compute the image brightness, and save
the brightness in a list called `results`. Then, we add the list as
a new column in the dataset. We will divide the brightness by 255 so that the
values range from 0 (completely black) to 1 (completely white).

In [ ]:
results = []

for row in photos.iter_rows(named=True):
    img = DSImage.read(row["filepath"])
    results.append(img.mean() / 255)

photos = (
    photos
    .with_columns(brightness = pl.Series(results))
)

Now that we have added the annotation of the image brightness to each of the
rows of the data, we can arrange the photographs from the brightest to the
darkest using the `sort` method.

In [ ]:
(
    photos
    .sort(c.brightness, descending=True)
    .select(c.nara_id, c.photographer, c.state, c.brightness, c.title)
)

The theory of distant viewing tells us that the creation of annotations, while
a necessary step in computational analysis of images, is both destructive and
subjective. So, before continuing to an aggregative analysis, it is useful to
connect the annotations back to the images by actually looking at some of the
photographs. One way to do that is to look at photographs with extreme values. In the
code below we show the twelve images that have been assigned the highest
brightness values.

In [ ]:
(
    photos
    .sort(c.brightness, descending=True)
    .head(12)
    .pipe(DSImage.plot_image_grid, label_name="state", ncol=6)
)

It's similarly useful to look at the darkest images in the dataset. To do that
we sort the data in the other direction.

In [ ]:
(
    photos
    .sort(c.brightness)
    .head(12)
    .pipe(DSImage.plot_image_grid, label_name="state", ncol=6)
)

After looking at some example photographs, how do you feel about the annotation's
ability to capture brightness? At least at the extremes, what feature(s)
of the photographs seem to best explain/predict the brightness of the image?

Now that we have some understanding of how the annotation works, and hopefully
some confidence that it corresponds with some meaningful quantity, let's do
some aggregative analysis with the annotations. In the code below we group our
dataset by `region` and look at the mean
average brightness of all photographs from that region, along with the number
of photographs.

In [ ]:
(
    photos
    .group_by(c.region)
    .agg(
        brightness_avg = c.brightness.mean(),
        count = pl.len()
    )
    .sort(c.brightness_avg)
)

An average hides how much the photographs within each region differ from one
another. A plot can show this. The code below draws a boxplot of the
brightness of the photographs in each region. The line in the middle of each
box is the median brightness, and the box covers the middle half of the
photographs.

In [ ]:
(
    photos
    .sort(c.brightness.median().over(c.region))
    .ggplot(aes(c.region, c.brightness))
    .geom_boxplot()
    .coord_flip()
)

How would you characterize the pattern here? Is there a meaningful pattern?
Keep in mind that we have only 100 photographs. The differences between the
regions are small compared to the differences among the photographs within
each one.

## 1.6 Saturation and Chroma

We have already seen that if we do a careful analysis, we can do quite a bit of
work with a relatively straightforward annotation based on image brightness. Our goal
though is to understand the use of color more broadly, which
will require creating additional annotations that capture other aspects of
color. One way to do this is to first convert the raw pixel intensities
into an alternative color space.

The RGB representation of pixels by the
amount of red, green, and blue light needed to create the color at a specific
point in space comes from the low-level engineering needs to image capture and
display. As we have already seen, it is not a particularly meaningful way to
think about our perception of color. Fortunately, there are other ways to
represent color that more closely align with human perception and understanding.
In order to understand how this works, let's re-load the photograph taken at
Dead Horse Point. Recall that above we located a specific pixel in the tan
colored rock. It has a RGB representation of the following:

In [ ]:
img = DSImage.read("documerica/545607.jpg")
img[560, 100, :]

We can convert the RGB format into an HSV format using the `cv2.cvtColor`
function by specifying the type of color space transformation (`COLOR_RGB2HSV`)
to use as a second argument.

In [ ]:
img_hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
img_hsv.shape

Notice that the shape of the output is exactly the same as the original image.
The rows and columns still correspond to the same locations as the RGB model;
it is only the triple of numbers at that location that have changed. Let's see
what our pixel from the rock looks like in this new color space.

In [ ]:
img_hsv[560, 100, :]

The first component corresponds to the **hue** of the
pixel, with this number corresponding to the color orange. Hue is a bit complex
and we will further investigate how it works in the next section. The second
number is the **saturation**, which represents how rich the color is, on a
scale from 0 to 255. A light
pastel, such as a pale pink, will have a low saturation. Here, we see that the
saturation is fairly low. Finally, the **value** is an alternative
representation of brightness, also on a scale from 0 to 255.

Let's now focus on the saturation of the photographs and do a similar analysis to
the ones above with brightness. To more closely follow the analysis
in the book, we will compute the related quantity called **chroma** instead of working with saturation directly. This can be computed by
multiplying the saturation by the value, after dividing each by 255 so that
they range from 0 to 1. We used this quantity because it more
closely aligns with the idea of the richness of color that we wanted to capture.
A dark color can have a high saturation and still look nearly black. Only a
"pure" color, like what you would see on a color wheel, has a chroma of 1.

Now, let's cycle through the photographs and add the average chroma value to each of
them just as we did with the image brightness.

In [ ]:
results = []

for row in photos.iter_rows(named=True):
    img = DSImage.read(row["filepath"])
    img_hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    chroma = (img_hsv[:, :, 1] / 255) * (img_hsv[:, :, 2] / 255)
    results.append(chroma.mean())

photos = (
    photos
    .with_columns(chroma = pl.Series(results))
)

Now, let's once again look at some of the photographs that correspond to
extreme values. It is always an important step when working with new annotations
to go back to the original images, and actually look at them to see how the
numeric representation of the image corresponds to our own viewing
and interpretation. Here are the twelve photographs with the highest average
chroma.

In [ ]:
(
    photos
    .sort(c.chroma, descending=True)
    .head(12)
    .pipe(DSImage.plot_image_grid, label_name="state", ncol=6)
)

And here are the twelve photographs with the lowest average chroma.

In [ ]:
(
    photos
    .sort(c.chroma)
    .head(12)
    .pipe(DSImage.plot_image_grid, label_name="state", ncol=6)
)

We now have two annotations for each photograph, and we can look at how they
relate to one another. The code below draws a scatter plot with one point for
each photograph, with the brightness along the horizontal axis and the chroma
along the vertical axis. The points are colored by region.

In [ ]:
(
    photos
    .ggplot(aes(c.brightness, c.chroma))
    .geom_point(aes(color=c.region))
)

Take some time to look at the results. Notice that even the most colorful
photographs have a low average chroma, far from the maximum value of 1. Does
anything seem either (i) particularly surprising or (ii) particularly
unsurprising? Both of these are useful observations for understanding the
connection between the photographs and the annotations that we have made from
them.

## 1.7 Dominant Color

Having looked at the brightness/value and saturation/chroma of the photographs,
we now look to the third element of color, know as **hue**. To get started,
let's once more take the photograph from Dead Horse Point and compute its HSV
coordinates just as we did in the previous section. This time, we will put the
pixel data into a table that has one row for each pixel and one column for
each of the three components. This is just some
re-arranging to make the rest of the analysis easier.

In [ ]:
img = DSImage.read("documerica/545607.jpg")
img_hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)

pixels = pl.DataFrame({
    "hue": img_hsv[:, :, 0].flatten(),
    "saturation": img_hsv[:, :, 1].flatten(),
    "value": img_hsv[:, :, 2].flatten()
})
pixels

The hue is a number that indicates what we would colloqually
call "color". Unlike brightness, saturation, chroma, and value, these numbers
are best thought of being arranged in a circle. The library that we are using
gives the hue as a number between 0 and 179. A value of `0` corresponds with
red, `30` with yellow, `60` with green, `90`
with cyan, and `120` with blue. Larger values wrap back through purple and
link back into red. So, the hues `1` and `178` are actually quite similar.

To understand a bit better, let's look at a histogram showing the distribution
of the hues in the image. We need to be careful, though, because our
interpretation of hue is only applicable when the color is sufficently rich
and bright.
If the saturation or the value is small, there is little color to show anyway
and the
differences between hues may be difficult or impossible to differentate. In
the code below, we show the distribution of hues for the pixels that have
both a saturation and a value of at least 50.

In [ ]:
(
    pixels
    .filter((c.saturation >= 50) & (c.value >= 50))
    .ggplot(aes(c.hue))
    .geom_histogram(binwidth=2, fill="white", color="black")
)

Most of the pixels in the histogram have a hue between `0` and about `20`;
these correspond to the reds and oranges of the rock, which takes up a lot of
space in the image. The values at the far right of the plot, near `170`, are
also reds, leaning towards magenta, since the hues wrap around in a circle. The small group of values
near `100` is associated with the blue in the image, mainly the
photographer's jeans. Notice that most of the sky does not appear in the plot
at all, because it is too pale to pass our filter.

Taking averages of hues does not in general provide meaningful summaries. For
an extreme example, if we take the average of two shades of red that have hues
of `178` and `2`, this would yield a hue of `90`, cyan, a color directly
between green and blue. Alternatively, we can create an annotation of hue by
breaking the range of hues up into standard color names and then count the
proportion of each photograph that corresponds with each color. The function
`DSImage.compute_colors` does this, returning the percentage of the pixels in
an image that fall into each of eight named colors. The pixels with too little
saturation or value to have a meaningful hue are counted as `neutral`.

In [ ]:
DSImage.compute_colors(img_hsv)

We see that about 18 percent of this photograph corresponds to hues that
have been categorized as "orange" and about 13 percent to "red", with only a
small amount of any other color. About 60 percent of the pixels are counted
as neutral.

Now that we have seen how to do this with a single image, let's cycle through
all of the photographs and compute the percentage
of the image corresponding to each color for each of them. Each result is a
set of nine numbers, which we turn into a table and place alongside the
columns that we already have.

In [ ]:
results = []

for row in photos.iter_rows(named=True):
    img = DSImage.read(row["filepath"])
    img_hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    results.append(DSImage.compute_colors(img_hsv))

results = pl.DataFrame(results)
photos = pl.concat([photos, results], how="horizontal")
photos

As with the other two annotations, we can start by looking at the photographs
that have the largest amount of a given color. Here, for example,
is the code to show the images with the largest amount of orange. Try to change the
code to see other colors.

In [ ]:
(
    photos
    .sort(c.orange, descending=True)
    .head(12)
    .pipe(DSImage.plot_image_grid, label_name="state", ncol=6)
)

There are a number of different ways to summarize these percentages. One is to
associate each photograph with a dominant color
corresponding to the color name that is most strongly represented in it.
To do this, we first rearrange the data so that there is one row for each
combination of a photograph and a color. Then, we sort the rows by the
percentage and keep the first row for each photograph.

In [ ]:
colors = [c.red, c.orange, c.yellow, c.green, c.cyan, c.blue, c.purple, c.magenta]

dominant = (
    photos
    .unpivot(
        index=[c.nara_id, c.filepath, c.region],
        on=colors,
        variable_name=c.color,
        value_name=c.percent
    )
    .sort(c.percent, descending=True)
    .group_by(c.nara_id)
    .head(1)
)
dominant

Now, let's do some analysis with these annotations. We can start by counting
the number of photographs that have each color as their dominant color.

In [ ]:
(
    dominant
    .group_by(c.color)
    .agg(count = pl.len())
    .sort(c.count)
    .ggplot(aes(c.color, c.count))
    .geom_col()
    .coord_flip()
)

Blue is the dominant color of about half of the photographs, followed by red
and orange. Very few have green as a dominant color, even though many of the
photographs show trees, fields, and other vegetation. Look back at the grids of
images above: the grass and the trees are often very dark or have a blue cast
rather than a bright green. The annotation describes the digital image that we
have, which depends on the film that was used, on how the film has aged, and on
how it was scanned, as much as on the scene that was in front of the camera.

Another way to summarize the colors is to take the average percentage of each
color across the photographs from each region.

In [ ]:
(
    photos
    .group_by(c.region)
    .agg(
        blue = c.blue.mean(),
        red = c.red.mean(),
        orange = c.orange.mean(),
        green = c.green.mean(),
        neutral = c.neutral.mean()
    )
    .sort(c.orange, descending=True)
)

Do you see any interesting patterns in the data above? The photographs from the
West have the largest amounts of red and orange, which in our sample include
the red rock deserts of Utah and Arizona. The amount of green is very small
everywhere, but it is larger in the South and the Northeast than in the West
and the Midwest. Try to change the code to look at other colors and see if any
other patterns arise.

## 1.8 Face Detection

When introducing the ideas behind distant viewing and the computational analysis
of large collections of digital images, we like to start with annotations that
capture color. There's a tangible connection between elements such as
brightness, saturation, and hue that we can instantly connect to the way that
digital images are created and stored. At the same time, color is not at all
simple. In many ways, it is a particularly difficult annotation to work with
because there are so many different ways to count, bucket, and summarize the
results. Understanding the connection between the raw pixel intensites, the
resulting annoations, and the semantic gap between the two is key to
understanding the challenges and possibilites of working with collections of
digital images, and distant viewing.

Of course, many applications of distant viewing will want to integrate other
kinds of annotations, many of which require using more advanced machine learning
techniques such as neural networks. In this final
section of analysis, we will show how to use a face detection model to add
an additional annotation to our dataset using the distant viewing
toolkit (**dvt**).

The distant viewing toolkit simplifes the process of applying several common
computer visional algorithms to still and moving images. Most of the difficult
work of standardizing the inputs, loading models, and getting all of the
results into the same format is taken care of in dvt. The toolkit is built
around a single function called `annotate`. We give it an image, a folder of
images, or a video file, and name one or more *annotators* that we want to run.
Each annotator creates a table of structured data. In the code below we run
the `face` annotator over the folder that holds all of our photographs.

Why might we be interested in detecting faces in these photographs?
Which questions could we ask?

In [ ]:
tables = dvt.annotate("documerica", annotators=["face"])

The result is a set of tables. The one called `face` has one row for each
face that was found in each photograph. We will keep the columns that give the
name of the image file (`path`), a number for each face within the image
(`face_id`), a score between 0 and 1 describing how confident the algorithm
is that it has found a face (`det_score`), and the location of a box around
the face, in pixels from the upper-left hand corner of the image.

In [ ]:
face = (
    tables["face"]
    .select(
        c.path, c.face_id, c.det_score,
        c.bbox_x1, c.bbox_y1, c.bbox_x2, c.bbox_y2
    )
)
face

This structured version of the detected faces is what we will use for our
analysis. First, though, let's look at what the algorithm has found in one of
the photographs. In the code below we load a photograph taken in Orick,
California, select the rows of the `face` table for that image, and draw each
of the boxes onto the image.

In [ ]:
img = DSImage.read("documerica/543043.jpg")

here = (
    face
    .filter(c.path == "543043.jpg")
)

for row in here.iter_rows(named=True):
    top_left = (int(row["bbox_x1"]), int(row["bbox_y1"]))
    bottom_right = (int(row["bbox_x2"]), int(row["bbox_y2"]))
    cv2.rectangle(img, top_left, bottom_right, (0, 255, 0), 2)

DSImage.show(img)

This kind of visualization is useful for the qualatitive
analysis and assessment of how well the algorithm works on our data. We see
in this example that the algorithm has found the faces of the people in the
photograph.

The annotations above provide one row for each detected face in one of the
photographs. In order to analyze these annotations, we need to aggregate
them so that we have a single summary value for each photograph. One way to do
that is to filter the faces to only those above a threshold
score and then count how many faces are present in each photograph. Then, we
can add this count back into the `photos` table. We need to be careful
about doing this so as to make sure that we are not missing the photographs
that have zero faces. The sequence of steps below puts all of
these together. We use a threshold of `0.7`.

In [ ]:
face_count = (
    face
    .filter(c.det_score > 0.7)
    .group_by(c.path)
    .agg(num_face = pl.len())
    .with_columns(filepath = "documerica/" + c.path)
)

photos = (
    photos
    .join(face_count.select(c.filepath, c.num_face), on=c.filepath, how="left")
    .with_columns(num_face = c.num_face.fill_null(0))
)

(
    photos
    .sort(c.num_face, descending=True)
    .select(c.nara_id, c.num_face, c.state, c.title)
)

As we have done with the color annotations, it is a good idea to look at some
of the results. Here are the twelve photographs with the largest number of
detected faces, labelled with the number that were found.

In [ ]:
(
    photos
    .sort(c.num_face, descending=True)
    .head(12)
    .pipe(DSImage.plot_image_grid, label_name="num_face", ncol=6)
)

Does the number of detected faces match the number of people that you can see
in each photograph? In many of these images there are people whose faces are
turned away from the camera, are in shadow, or are too small to find. The
annotation counts faces that the algorithm can find, which is not the same
thing as counting people.

Most of the photographs in our sample have no detected faces at all. As a
final analysis, we will compute the proportion of photographs in each region
that have at least one detected face.

In [ ]:
(
    photos
    .group_by(c.region)
    .agg(
        face_present = (c.num_face > 0).mean(),
        count = pl.len()
    )
    .sort(c.face_present, descending=True)
)

About a third of the photographs from the South have at least one detected
face, compared with one in ten of the photographs from the Northeast. With so
few photographs this is a pattern to investigate further rather than a
conclusion. Can you think of any hypotheses about what might be behind it? Do
the photographs with people in them differ in their brightness or color from
the ones without?

In the book, we decided to focus on one type of annotation in each chapter. Here, we begin to demonstrate how layering types of annotations begins to add new areas of exploration and to nuance our computaitonal analysis of images.

## 1.9 Conclusions and Next Steps

The tutorial has covered a lot of material. We've introduced the basics of
running Python and an understanding of how digital images are represented
as arrays of pixel intensites. From there, we motived the theoretical stakes
of distant viewing. Then, we put this into practice by generating increasingly
complex annotations based on our understanding of color and, finally, by using
a neural-network based machine learning algorithm to detected faces present in
each of the photographs. Throughout, we have tried to connect each of our
annotations back to the images themselves by looking at them.

There are many directions to explore after following this notebook. If you want
to see a complete analysis of the use of color in a larger collection, including the important
steps of connecting our observations to existing archival and scholarly sources,
we suggest checking out Chapter 3 of our *Distant Viewing* book, which is
available on our [website](https://www.distantviewing.org/book/). If you are new to programming, learning
a [bit more Python](https://wiki.python.org/moin/BeginnersGuide)
from core principles is a good start. If your interests including moving
images, we have a [follow-up notebook](https://colab.research.google.com/github/distant-viewing/dvt/blob/main/tutorials/Distant_Viewing_Tutorial_2_Shots_and_Faces_in_Moving_Images.ipynb) that works through an
example using a clip from a television show.
Otherwise, we suggest checking out the other annotations available in the
[distant viewing toolkit](https://github.com/distant-viewing/dvt)
and applying them to your own collections.


Thank you for taking time to engage with us, and we welcome feedback!
- Lauren and Taylor | [Distant Viewing Lab](https://distantviewing.org)